In [1]:
import fitz
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os


/home/mouad/anaconda3/envs/rag-assistant/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
# 1. Extraction + Chunking, page par page, avec tracking de la source
pdf_filename = os.listdir("../data")[0]
doc = fitz.open(f"../data/{pdf_filename}")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = []      # les textes des chunks (comme avant)
metadatas = []   # NOUVEAU : info source pour chaque chunk (même ordre que chunks)

for page_num, page in enumerate(doc):
    page_text = page.get_text()
    page_chunks = splitter.split_text(page_text)  # on chunk CETTE page uniquement

    for chunk in page_chunks:
        chunks.append(chunk)
        metadatas.append({
            "source": pdf_filename,
            "page": page_num + 1  # +1 car PyMuPDF compte les pages à partir de 0
        })

doc.close()

print(f"Nombre de chunks : {len(chunks)}")
print(f"Nombre de metadata : {len(metadatas)}")  # doit être EXACTEMENT le même nombre
print("\n--- Chunk 1 ---")
print(chunks[0])
print(metadatas[0])

Nombre de chunks : 291
Nombre de metadata : 291

--- Chunk 1 ---
SOUKAINA BOUAROUROU
soukaina.bouarourou@fsr.um5.ac.ma
CLOUD COMPUTING ET VIRTUALISATION
ANNÉE UNIVERSITAIRE 
2025 - 2026
Master  Ingénierie Informatique et Sécurité Systèmes
{'source': 'cloud_course.pdf', 'page': 1}


In [17]:
import chromadb
import hashlib

client_db = chromadb.PersistentClient(path="../vectorstore")
collection = client_db.get_or_create_collection(name="mon_cours")




# Étape 1 — Dédupliquer AVANT d'envoyer à Chroma
seen_ids = set()
unique_chunks = []
unique_metadatas = []
unique_ids = []

for chunk, meta in zip(chunks, metadatas):
    chunk_id = hashlib.md5(chunk.encode()).hexdigest()
    if chunk_id not in seen_ids:
        seen_ids.add(chunk_id)
        unique_chunks.append(chunk)
        unique_metadatas.append(meta)
        unique_ids.append(chunk_id)

print(f"Avant dédup : {len(chunks)} chunks — Après dédup : {len(unique_chunks)} chunks")

# Étape 2 — Encoder et indexer seulement les chunks uniques
embeddings = model.encode(unique_chunks).tolist()

collection.upsert(
    documents=unique_chunks,
    embeddings=embeddings,
    ids=unique_ids,
    metadatas=unique_metadatas
)

print(f"{len(unique_chunks)} chunks indexés (total dans la collection : {collection.count()})")

Avant dédup : 291 chunks — Après dédup : 270 chunks
270 chunks indexés (total dans la collection : 270)


In [29]:
question = "les avantages du cloud computing ?"
question_embedding = model.encode([question]).tolist()

results = collection.query(
    query_embeddings=question_embedding,
    n_results=3
)

for i in range(len(results["documents"][0])):
    doc = results["documents"][0][i]
    meta = results["metadatas"][0][i]
    print(f"--- Résultat {i+1} (source: {meta['source']}, page {meta['page']}) ---")
    print(doc)
    print()

--- Résultat 1 (source: cloud_course.pdf, page 180) ---
Avantages et inconvénients du cloud
Avantages :
Recentrage sur le cœur de métier :
•
Le recours au cloud computing permet de décharger les équipes informatique des
entreprises, qui ont alors plus de disponibilité pour des activités à haute
valeur
ajoutée.
•
La maintenance, la sécurisation et les évolutions des services étant à la charge
exclusive du prestataire, dont c'est généralement le cœur de métier, celles-ci ont

--- Résultat 2 (source: cloud_course.pdf, page 178) ---
Avantages et inconvénients du cloud
32
Avantages :
Réduction des coûts
•
Les caractéristiques du cloud computing intéressantes pour les entreprises sont
la réduction du coût total de possession des systèmes informatiques, la
facilité d'augmenter ou de diminuer les ressources.
•
Le cloud computing peut permettre d'effectuer des économies, notamment
grâce à la mutualisation des services sur un grand nombre de clients. Certains

--- Résultat 3 (source: cloud_cours

In [19]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2663.66it/s]


In [9]:
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()  # charge le .env (project root, donc si t'es dans notebooks/ faut peut-être load_dotenv("../.env"))

client_groq = Groq(api_key=os.getenv("GROQ_API_KEY")) 

In [30]:
question = "les avantages du cloud computing ?"
question_embedding = model.encode([question]).tolist()

results = collection.query(
    query_embeddings=question_embedding,
    n_results=5
)

context_parts = []
for i in range(len(results["documents"][0])):
    doc = results["documents"][0][i]
    meta = results["metadatas"][0][i]
    context_parts.append(f"[Source: {meta['source']}, page {meta['page']}]\n{doc}")

context = "\n\n".join(context_parts)

prompt = f"""Tu es un assistant qui répond aux questions en te basant UNIQUEMENT sur le contexte fourni ci-dessous, extrait d'un cours.

Contexte :
{context}

Question : {question}

Réponds de manière claire et concise, en te basant uniquement sur le contexte. Indique la page source à la fin de ta réponse (ex: "Source : page X"). Si l'information n'est pas dans le contexte, dis-le clairement plutôt que d'inventer."""

response = client_groq.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.2
)

print(response.choices[0].message.content)

Les avantages du cloud computing sont :

- Recentrage sur le cœur de métier (Source : page 180)
- Réduction des coûts (Source : page 178)
- Facilité d'augmenter ou de diminuer les ressources (Source : page 178)
- Mutualisation des services sur un grand nombre de clients (Source : page 178)
- Libération de la nécessité de gérer des serveurs physiques (Source : page 142)
- Possibilité d'effectuer des économies (Source : page 178)

Source : page 142, 178, 180


In [1]:
# === SETUP (une seule fois) ===
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import os
from dotenv import load_dotenv

import sys
sys.path.append("../src")

from extraction_and_chunking import extract_and_chunk
from dep_index import index_chunks
from retrieve_chunks import retrieve_chunks
from build_promt import build_prompt
from generate_response import generate_answer

load_dotenv("../.env")

model = SentenceTransformer("all-MiniLM-L6-v2")
client_db = chromadb.PersistentClient(path="../vectorstore")
collection = client_db.get_or_create_collection(name="mon_cours")
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# === PIPELINE COMPLET ===
chunks, metadatas = extract_and_chunk("../data/cloud_course.pdf")
nb_indexed = index_chunks(chunks, metadatas, collection, model)
print(f"{nb_indexed} chunks indexés")

question = "quels sont les avantages du cloud computing ?"
retrieved_chunks, retrieved_metadatas = retrieve_chunks(question, collection, model, n_results=5)
prompt = build_prompt(question, retrieved_chunks, retrieved_metadatas)
answer = generate_answer(prompt, groq_client)

print(answer)

/home/mouad/anaconda3/envs/rag-assistant/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2043.61it/s]


270 chunks indexés
Les principaux avantages du cloud computing, tels que présentés dans le cours, sont :

- **Recentrage sur le cœur de métier** : le cloud décharge les équipes informatiques, leur permettant de se concentrer sur des activités à forte valeur ajoutée.  
- **Maintenance, sécurisation et évolution prises en charge par le prestataire** : ces tâches sont gérées par le fournisseur, qui est spécialisé dans ce domaine.  
- **Réduction du coût total de possession** : le cloud diminue les dépenses liées aux infrastructures informatiques.  
- **Flexibilité de montée/descente de capacité** : il est facile d’ajuster les ressources en fonction des besoins.  
- **Économies grâce à la mutualisation** : les services partagés entre de nombreux clients permettent de réaliser des économies d’échelle.

**Source : pages 180 et 178**
